# CLEANING — Berlin Marathon Raw Data Processing\n\n**Input:** `../data/Dataset_Berlin_Marathon_1999-2025_original.csv` (880,779 rows)  \n**Output:** `../data/berlin_marathon_clean.parquet`\n\nPipeline:\n1. Load raw CSV (semicolon separator)\n2. Rename columns for consistency\n3. Convert time strings (HH:MM:SS) to seconds\n4. Filter invalid/missing records (documented at each step)\n5. Compute 9 segment paces (min/km) and pacing CV\n6. Save clean parquet

In [ ]:
import pandas as pd
import numpy as np

# --- 1. Load raw data ---
from pathlib import Path

DATA_PATH = Path('../data/Dataset_Berlin_Marathon_1999-2025_original.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError(
        'Dataset not found. Download from Zenodo and place in data/ directory. '
        'See data/README.md for instructions.'
    )
raw = pd.read_csv(DATA_PATH, sep=';', low_memory=False)
N_RAW = len(raw)
print(f"Raw records loaded: {N_RAW:,}")
print(f"Columns: {list(raw.columns)}")
print(f"Years: {raw['year'].min()} – {raw['year'].max()}")
raw.head(3)

In [ ]:
# --- 2. Rename columns ---
df = raw.rename(columns={
    'sex': 'gender',
    'nation': 'country',
    'age_category': 'age_group',
    'final': 'net_time',
})

SPLIT_COLS = ['5km', '10km', '15km', '20km', 'half', '25km', '30km', '35km', '40km']
ALL_TIME_COLS = SPLIT_COLS + ['net_time']

print(f"Columns after rename: {list(df.columns)}")

In [ ]:
# --- 3. Convert time strings to seconds ---
def time_to_seconds(s):
    """Convert HH:MM:SS string to total seconds. Returns NaN for invalid values."""
    if pd.isna(s) or str(s).strip() in ('', '-', 'DNF', 'DNS', 'DSQ'):
        return np.nan
    try:
        parts = str(s).strip().split(':')
        if len(parts) == 3:
            h, m, sec = int(parts[0]), int(parts[1]), int(parts[2])
            total = h * 3600 + m * 60 + sec
            return total if total > 0 else np.nan  # 00:00:00 = missing
        return np.nan
    except (ValueError, IndexError):
        return np.nan

for col in ALL_TIME_COLS:
    df[col + '_sec'] = df[col].apply(time_to_seconds)

# Suffix-free column names for seconds versions
SEC_SPLIT_COLS = [c + '_sec' for c in SPLIT_COLS]
print("Time conversion complete.")
print(f"Sample — 5km_sec: {df['5km_sec'].iloc[0]:.0f}s, net_time_sec: {df['net_time_sec'].iloc[0]:.0f}s")

## Exclusion Steps (CONSORT-style)\n\nEach filter is applied sequentially, with counts at each step.

In [ ]:
# --- 4. Sequential exclusion filters ---
consort = [f"Raw records: {N_RAW:,}"]

# 4a. Missing net_time (DNF/DNS/invalid)
mask_net = df['net_time_sec'].notna()
n_before = len(df)
df = df[mask_net].copy()
excluded = n_before - len(df)
consort.append(f"Excluded missing net_time: {excluded:,} → remaining: {len(df):,}")

# 4b. Missing half-marathon split
mask_half = df['half_sec'].notna()
n_before = len(df)
df = df[mask_half].copy()
excluded = n_before - len(df)
consort.append(f"Excluded missing half split: {excluded:,} → remaining: {len(df):,}")

# 4c. Biologically implausible finish times (< 1:59:00 or > 6:15:00)
MIN_TIME = 1 * 3600 + 59 * 60   # 7140s
MAX_TIME = 6 * 3600 + 15 * 60   # 22500s
mask_plausible = (df['net_time_sec'] >= MIN_TIME) & (df['net_time_sec'] <= MAX_TIME)
n_before = len(df)
df = df[mask_plausible].copy()
excluded = n_before - len(df)
consort.append(f"Excluded implausible finish time (<1:59 or >6:15): {excluded:,} → remaining: {len(df):,}")

# 4d. Standardize gender
df['gender'] = df['gender'].astype(str).str.strip().str.upper()
# Map known variants
gender_map = {'M': 'M', 'W': 'F', 'F': 'F', 'MALE': 'M', 'FEMALE': 'F',
              'MEN': 'M', 'WOMEN': 'F', 'MÄNNLICH': 'M', 'WEIBLICH': 'F'}
df['gender'] = df['gender'].map(gender_map)
mask_gender = df['gender'].notna()
n_before = len(df)
df = df[mask_gender].copy()
excluded = n_before - len(df)
consort.append(f"Excluded unmapped gender: {excluded:,} → remaining: {len(df):,}")

# 4e. Missing any of the 9 split checkpoints (needed for segment pace calculation)
mask_all_splits = df[SEC_SPLIT_COLS].notna().all(axis=1)
n_before = len(df)
df_full = df[mask_all_splits].copy()
excluded = n_before - len(df_full)
consort.append(f"Excluded missing any split checkpoint: {excluded:,} → remaining: {len(df_full):,}")

# 4f. Monotonicity check — split times must be strictly increasing
split_arr = df_full[SEC_SPLIT_COLS].values
diffs = np.diff(split_arr, axis=1)
# Also check first split > 0 and last split < net_time
first_ok = split_arr[:, 0] > 0
last_ok = split_arr[:, -1] < df_full['net_time_sec'].values
monotonic = (diffs > 0).all(axis=1) & first_ok & last_ok
n_before = len(df_full)
df_full = df_full[monotonic].copy()
excluded = n_before - len(df_full)
consort.append(f"Excluded non-monotonic splits: {excluded:,} → remaining: {len(df_full):,}")

N_CLEAN = len(df_full)
consort.append(f"\nFinal clean dataset: {N_CLEAN:,} ({N_CLEAN/N_RAW*100:.1f}% of raw)")

print("\n=== CONSORT Flow ===")
for line in consort:
    print(line)

In [ ]:
# --- 5. Compute 9 segment paces (min/km) ---
# Segment distances in km
SEGMENTS = [
    ('pace_05k',    0,         5.0,    '5km_sec',   None),       # 0 to 5km
    ('pace_510k',   5.0,      10.0,    '10km_sec',  '5km_sec'),
    ('pace_1015k', 10.0,      15.0,    '15km_sec',  '10km_sec'),
    ('pace_1520k', 15.0,      20.0,    '20km_sec',  '15km_sec'),
    ('pace_2025k', 20.0,      25.0,    '25km_sec',  '20km_sec'),
    ('pace_2530k', 25.0,      30.0,    '30km_sec',  '25km_sec'),
    ('pace_3035k', 30.0,      35.0,    '35km_sec',  '30km_sec'),
    ('pace_3540k', 35.0,      40.0,    '40km_sec',  '35km_sec'),
    ('pace_40End', 40.0,      42.195,  'net_time_sec', '40km_sec'),
]

PACE_COLS = []
for name, start_km, end_km, end_col, start_col in SEGMENTS:
    dist = end_km - start_km
    if start_col is None:
        elapsed = df_full[end_col]
    else:
        elapsed = df_full[end_col] - df_full[start_col]
    df_full[name] = (elapsed / 60) / dist  # min/km
    PACE_COLS.append(name)

# Filter extreme paces (< 2.5 or > 10 min/km in any segment)
pace_arr = df_full[PACE_COLS].values
valid_paces = ((pace_arr >= 2.5) & (pace_arr <= 10.0)).all(axis=1)
n_before = len(df_full)
df_full = df_full[valid_paces].copy()
excluded = n_before - len(df_full)
print(f"Excluded extreme segment paces: {excluded:,} → remaining: {len(df_full):,}")

# --- 6. Compute pacing CV ---
pace_values = df_full[PACE_COLS].values
df_full['pacing_cv'] = (pace_values.std(axis=1) / pace_values.mean(axis=1)) * 100

print(f"\nSegment paces and CV computed.")
print(f"Pacing CV — mean: {df_full['pacing_cv'].mean():.2f}%, median: {df_full['pacing_cv'].median():.2f}%")
df_full[PACE_COLS + ['pacing_cv']].describe().round(3)

In [ ]:
# --- 7. Select final columns and save ---
KEEP_COLS = [
    'year', 'gender', 'name', 'country', 'starting_num', 'age_group',
    # Split times in seconds
    '5km_sec', '10km_sec', '15km_sec', '20km_sec', 'half_sec',
    '25km_sec', '30km_sec', '35km_sec', '40km_sec', 'net_time_sec',
    # Segment paces (min/km)
] + PACE_COLS + ['pacing_cv']

out = df_full[KEEP_COLS].reset_index(drop=True)

# Optimize dtypes
out['year'] = out['year'].astype('int16')
out['gender'] = out['gender'].astype('category')
out['country'] = out['country'].astype('category')
out['age_group'] = out['age_group'].astype('category')

out.to_parquet('../data/berlin_marathon_clean.parquet', index=False)

print(f"Saved: ../data/berlin_marathon_clean.parquet")
print(f"Final shape: {out.shape}")
print(f"Gender distribution: {out['gender'].value_counts().to_dict()}")
print(f"Year range: {out['year'].min()} – {out['year'].max()}")
print(f"\nMemory usage: {out.memory_usage(deep=True).sum() / 1e6:.1f} MB")